# Evaluation

Today, we. are going to test our angle-based rep counter of videos of multiple reps of a certain exercise. We will measure two metrics, the average amount that our "model" is off by, and the accuracy, or how many times the model got it perfectly correct.

In this notebook, we are only testing the rep counter and it will be simple on videos of single reps strung together.

In [96]:
import numpy as np
import cv2 as cv
import mediapipe as mp
import random
import os
from tqdm import tqdm

In [97]:
# same repcounter that is in src/rep_counter.py

class RepCounter:

    def __init__(self, exercise: str):
        self.exercise = exercise
        BaseOptions = mp.tasks.BaseOptions
        self.PoseLandmarker = mp.tasks.vision.PoseLandmarker
        PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
        VisionRunningMode = mp.tasks.vision.RunningMode

        self.options = PoseLandmarkerOptions(
            base_options=BaseOptions(model_asset_path='/Users/ansh/Downloads/development/repquest/pose_landmarker_full.task'), # this has been modified
            running_mode=VisionRunningMode.VIDEO,
            num_poses=1,
            min_pose_detection_confidence=0.5,
            min_pose_presence_confidence=0.5,
            min_tracking_confidence=0.5,
            output_segmentation_masks=False,
        )

        self.rep_map = {
            "pushup": (60, 135),
            "squat": (87.5, 160),
            "lunge": (90, 140),
        }

        self.rep_map_kypts = {
            "pushup": [[15, 13, 11], [16, 14, 12]], # right points, left points
            "squat": [[23, 25, 27], [24, 26, 28]],
            "lunge": [[23, 25, 27], [24, 26, 28]],
        }

        if self.exercise not in self.rep_map.keys():
            raise ValueError("enter valid exercise")
        
        self.MIN_ANGLE, self.UPRIGHT_POS_ANGLE = self.rep_map[self.exercise]
        self.WAIT_FRAMES = 20

    def convert_landmarks(self, pose):
        data = []
        for landmark_list in pose.pose_landmarks:
            landmarks_array = np.array([
                [lm.x, lm.y, lm.z] for lm in landmark_list
            ])
            data.append(landmarks_array)
        return np.array(data)

    def calculate_angle(self, a, b, c):  # was missing `self`
        ba = a - b
        bc = c - b
        cos_a = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
        return np.degrees(np.arccos(np.clip(cos_a, -1, 1)))  # was np.clip(-1, 1)

    def count_reps(self, path, show_img: bool, return_pose=False):
        cap = cv.VideoCapture(path) # can be 0 or an actual path
        
        # cycle steps
        self.initial = False
        self.low = False
        self.back_up = False
        self.wait_frames_remaining = 0
        self.wait_over = True
        self.n_reps = 0
        frame_idx = 0

        pose = []

        with self.PoseLandmarker.create_from_options(self.options) as self.landmarker:
            while True:
                ret, frame = cap.read()
                if not ret: break
                frame = cv.flip(frame, 1)
                rgb_frame = cv.cvtColor(frame, cv.COLOR_BGR2RGB) # convert to rgb
                mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame) # convert to mp image
                timestamp_ms = int((frame_idx/30) * 1000) # get timestamp
                result = self.landmarker.detect_for_video(mp_img, timestamp_ms) # inference
                if not result.pose_landmarks:
                    frame_idx += 1
                    if show_img:
                        cv.imshow("frame", frame)
                        continue
                    continue
                landmark_arr = self.convert_landmarks(pose=result)
                
                if return_pose:
                    pose.append(landmark_arr)
                
                kypts = self.rep_map_kypts[self.exercise]
                right_pt1 = landmark_arr[0, kypts[0][0]]
                right_pt2 = landmark_arr[0, kypts[0][1]]
                right_pt3 = landmark_arr[0, kypts[0][2]]
                left_pt1  = landmark_arr[0, kypts[1][0]]
                left_pt2  = landmark_arr[0, kypts[1][1]]
                left_pt3  = landmark_arr[0, kypts[1][2]]
                
                right_angle = self.calculate_angle(right_pt1, right_pt2, right_pt3)
                left_angle = self.calculate_angle(left_pt1, left_pt2, left_pt3)

                if self.wait_over:
                    if not self.initial and not self.low and not self.back_up:
                        if right_angle > self.UPRIGHT_POS_ANGLE or left_angle > self.UPRIGHT_POS_ANGLE:
                            self.initial = True
                    elif self.initial and not self.low:
                        if right_angle < self.MIN_ANGLE or left_angle < self.MIN_ANGLE:
                            self.low = True
                    elif self.initial and self.low and not self.back_up:
                        if right_angle > self.UPRIGHT_POS_ANGLE or left_angle > self.UPRIGHT_POS_ANGLE:
                            self.back_up = True
                    if self.initial and self.low and self.back_up:
                        self.initial, self.low, self.back_up = False, False, False
                        self.wait_over = False
                        self.wait_frames_remaining = self.WAIT_FRAMES
                        self.n_reps += 1
                
                else:
                    self.wait_frames_remaining -= 1
                    if self.wait_frames_remaining == 0:
                        self.wait_over = True
                frame_idx += 1


                if show_img:
                    cv.imshow("frame", frame)
                if cv.waitKey(1) & 0xFF == ord('q'):
                    break
        
        if not return_pose:
            return self.n_reps
        else:
            return self.n_reps, pose

In [ ]:
def create_sample_and_test(exercise: str):
    GOOD_REP_WEIGHT = 3

    n_reps = random.randint(1, 20)
    n_counted_reps = 0
    workout_labels = []

    ROOT_DIR = f"/Users/ansh/Downloads/development/repquest/video_data/{exercise}"
    variations = os.listdir(ROOT_DIR)
    all_filenames = []
    for directory in variations:
        if directory == ".DS_Store":
            continue
        files = os.listdir(f"{ROOT_DIR}/{directory}")
        for file in files:
            if file != ".DS_Store":
                all_filenames.append(f"{directory}/{file}")
                workout_labels.append(directory)
    
    weights = [GOOD_REP_WEIGHT if "good" in f else 1 for f in all_filenames]

    os.makedirs("/Users/ansh/Downloads/development/repquest/temp_videos/", exist_ok=True)
    out = cv.VideoWriter(
        "/Users/ansh/Downloads/development/repquest/temp_videos/temp.mp4",
        cv.VideoWriter_fourcc(*"mp4v"),
        30,
        (1920, 1080),
    )

    for rep in range(n_reps):
        vid = random.choices(all_filenames, weights=weights, k=1)[0]
        if "good" in vid:
            n_counted_reps += 1
        
        cap = cv.VideoCapture(f"{ROOT_DIR}/{vid}")
        while True:
            ret, frame = cap.read()
            if not ret: break
            out.write(frame)
        cap.release()

    out.release()

    rep_counter = RepCounter(exercise=exercise)
    predicted_reps = rep_counter.count_reps(
        path="/Users/ansh/Downloads/development/repquest/temp_videos/temp.mp4",
        show_img=False,
        return_pose=False,
    )

    os.remove("/Users/ansh/Downloads/development/repquest/temp_videos/temp.mp4")

    return {
        "actual": n_counted_reps,
        "predicted": predicted_reps,
    }

In [99]:
n = 100
correct = 0
incorrect = 0
diff = 0

for i in tqdm(range(n), desc="Testing"):
    exercise = random.choice(["pushup", "squat", "lunge"])
    out = create_sample_and_test(exercise=exercise)
    actual = out["actual"]
    prediction = out["predicted"]

    if actual == prediction:
        correct += 1
    else:
        incorrect += 1
        diff += abs(actual - prediction)
    
    tqdm.write(f"[{i+1}/{n}] {exercise} | actual: {actual}, predicted: {prediction} | acc: {correct/(i+1):.0%}")

Testing:   0%|          | 0/100 [00:00<?, ?it/s]I0000 00:00:1781357374.411829 16410165 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4
W0000 00:00:1781357374.457185 16410168 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781357374.463415 16410175 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
Testing:   1%|          | 1/100 [00:23<38:23, 23.27s/it]

[1/100] lunge | actual: 4, predicted: 4 | acc: 100%


I0000 00:00:1781357399.867540 16411119 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4
W0000 00:00:1781357399.916556 16411125 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781357399.922339 16411123 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
Testing:   2%|▏         | 2/100 [01:00<51:41, 31.65s/it]

[2/100] squat | actual: 10, predicted: 10 | acc: 100%


I0000 00:00:1781357438.283681 16412493 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4
W0000 00:00:1781357438.328816 16412496 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781357438.335141 16412498 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
Testing:   3%|▎         | 3/100 [01:44<59:45, 36.96s/it]

...

[98/100] pushup | actual: 7, predicted: 7 | acc: 100%


I0000 00:00:1781360542.146021 16525403 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4
W0000 00:00:1781360542.191269 16525406 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781360542.197130 16525414 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
Testing:  99%|█████████▉| 99/100 [53:28<00:37, 37.43s/it]

[99/100] pushup | actual: 11, predicted: 11 | acc: 100%


I0000 00:00:1781360582.573096 16526601 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4
W0000 00:00:1781360582.619347 16526605 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781360582.625477 16526605 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
Testing: 100%|██████████| 100/100 [53:49<00:00, 32.29s/it]

[100/100] pushup | actual: 5, predicted: 5 | acc: 100%


In [100]:
# final
avg_diff = diff/n

print(f"\n\ncorrect: {correct}\nincorrect: {incorrect}\naverage diff: {avg_diff}\npercentage: {correct/n}")



correct: 100
incorrect: 0
average diff: 0.0
percentage: 1.0
